# Gold: Sensor Health

**Purpose**: Monitor the health of every sensor and flag maintenance needs: down detection, flatlines, coverage\
**Schedule**: Hourly, after the silver transformations\
**Source**: `airquality.silver` tables\
**Target**: `airquality.gold.sensor_health` (current state) and `airquality.gold.sensor_health_daily` (one row per sensor per day)

### 1.0: Configuration

In [ ]:
%run ./00_utils

### 2.0: Health Rules

In [ ]:
# Three independent health signals:
#   1. Silence:  hours since the last reading (STALE, then DOWN)
#   2. Flatline: the same value repeated over and over (stuck instrument)
#   3. Coverage: days with at least one reading in the last 7
STALE_AFTER_HOURS = 24     # a full day of silence: worth watching
DOWN_AFTER_HOURS = 72      # three days of silence: consider it down
FLATLINE_MIN_READINGS = 3  # need at least 3 readings to call a flatline
MIN_ACTIVE_DAYS_7D = 4     # fewer than 4 active days out of 7: poor coverage

# Time windows are anchored to the newest timestamp in the data (event time),
# not the wall clock: results depend on the data, not on when the job runs.
# If ingestion is delayed or paused, the health picture stays consistent
# instead of every sensor drifting to DOWN.
as_of = spark.sql(f"SELECT MAX(datetime_utc) FROM {CATALOG}.silver.measurements").collect()[0][0]
print(f"Evaluating sensor health as of: {as_of} UTC")

### 3.0: Current State (one row per sensor)

In [ ]:
# Built from silver, not from the fact table: sensors with zero measurements
# must be included, since "never reported" is itself a finding.
df_sensor_health = spark.sql(f"""
    WITH stats AS (
        -- Per-sensor activity aggregates, windows anchored to as_of
        SELECT
            sensors_id,
            MAX(datetime_utc) as last_reading_utc,
            COUNT(IF(datetime_utc >= TIMESTAMP'{as_of}' - INTERVAL 7 DAYS, 1, NULL)) as readings_last_7d,
            COUNT(DISTINCT IF(datetime_utc >= TIMESTAMP'{as_of}' - INTERVAL 7 DAYS, DATE(datetime_utc), NULL)) as active_days_last_7d,
            ROUND(AVG(IF(datetime_utc >= TIMESTAMP'{as_of}' - INTERVAL 7 DAYS, value, NULL)), 3) as avg_value_7d,
            COUNT(IF(datetime_utc >= TIMESTAMP'{as_of}' - INTERVAL 24 HOURS, 1, NULL)) as readings_24h,
            STDDEV(IF(datetime_utc >= TIMESTAMP'{as_of}' - INTERVAL 24 HOURS, value, NULL)) as stddev_24h
        FROM {CATALOG}.silver.measurements
        GROUP BY sensors_id
    ),

    health AS (
        -- Add pollutant and station context, denormalized on purpose:
        -- one flat table = one Looker Studio data source, no joins needed
        SELECT
            s.sensor_id,
            s.sensor_name,
            p.parameter_name,
            s.location_id,
            l.name as location_name,
            l.locality,
            l.latitude,
            l.longitude,
            st.last_reading_utc,
            TIMESTAMPDIFF(HOUR, st.last_reading_utc, TIMESTAMP'{as_of}') as hours_since_last_reading,
            COALESCE(st.readings_last_7d, 0) as readings_last_7d,
            COALESCE(st.active_days_last_7d, 0) as active_days_last_7d,
            st.avg_value_7d,
            -- Flatline: enough readings in the last 24h, all with the same value
            COALESCE(st.readings_24h >= {FLATLINE_MIN_READINGS} AND st.stddev_24h = 0, false) as is_flatline
        FROM {CATALOG}.silver.sensors s
        LEFT JOIN stats st ON s.sensor_id = st.sensors_id
        LEFT JOIN {CATALOG}.silver.parameters p ON s.parameter_id = p.parameter_id
        LEFT JOIN {CATALOG}.silver.locations l ON s.location_id = l.id
    )

    SELECT
        *,
        CASE
            WHEN last_reading_utc IS NULL THEN 'NEVER_REPORTED'
            WHEN hours_since_last_reading >= {DOWN_AFTER_HOURS} THEN 'DOWN'
            WHEN hours_since_last_reading >= {STALE_AFTER_HOURS} THEN 'STALE'
            ELSE 'HEALTHY'
        END as status,
        (last_reading_utc IS NULL
            OR hours_since_last_reading >= {DOWN_AFTER_HOURS}
            OR is_flatline
            OR active_days_last_7d < {MIN_ACTIVE_DAYS_7D}) as needs_maintenance,
        -- Human-readable reason: the table should read like a technician's work list
        CONCAT_WS('; ',
            IF(last_reading_utc IS NULL, 'never reported', NULL),
            IF(hours_since_last_reading >= {DOWN_AFTER_HOURS}, 'silent for {DOWN_AFTER_HOURS}h or more', NULL),
            IF(is_flatline, 'stuck on the same value for 24h', NULL),
            IF(last_reading_utc IS NOT NULL AND active_days_last_7d < {MIN_ACTIVE_DAYS_7D},
               CONCAT('reported only ', active_days_last_7d, ' of the last 7 days'), NULL)
        ) as maintenance_reason,
        TIMESTAMP'{as_of}' as as_of_utc
    FROM health
""")

# Snapshot semantics: current state, fully recomputable from silver,
# so the table is rebuilt from scratch on every run.
df_sensor_health.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.sensor_health")
print(f"sensor_health: {spark.table(f'{CATALOG}.gold.sensor_health').count()} sensors")

### 4.0: Daily History (one row per sensor per day)

In [ ]:
# Cross-join sensors with the observed calendar so silent days exist as rows:
# a day with zero readings is the signal, not a missing row. A running MAX
# window tracks, day by day, when each sensor last reported.
df_sensor_health_daily = spark.sql(f"""
    WITH bounds AS (
        SELECT MIN(DATE(datetime_utc)) as first_day, MAX(DATE(datetime_utc)) as last_day
        FROM {CATALOG}.silver.measurements
    ),

    calendar AS (
        SELECT explode(sequence(first_day, last_day, INTERVAL 1 DAY)) as date_id
        FROM bounds
    ),

    daily AS (
        -- What each sensor actually did on each day it reported
        SELECT
            sensors_id,
            DATE(datetime_utc) as date_id,
            COUNT(*) as readings,
            MAX(datetime_utc) as last_reading_utc,
            STDDEV(value) as value_stddev
        FROM {CATALOG}.silver.measurements
        GROUP BY sensors_id, DATE(datetime_utc)
    ),

    grid AS (
        -- Every sensor x every day in the observed range
        SELECT
            s.sensor_id,
            s.sensor_name,
            p.parameter_name,
            l.name as location_name,
            l.locality,
            c.date_id
        FROM {CATALOG}.silver.sensors s
        CROSS JOIN calendar c
        LEFT JOIN {CATALOG}.silver.parameters p ON s.parameter_id = p.parameter_id
        LEFT JOIN {CATALOG}.silver.locations l ON s.location_id = l.id
    ),

    filled AS (
        SELECT
            g.*,
            COALESCE(d.readings, 0) as readings,
            COALESCE(d.readings >= {FLATLINE_MIN_READINGS} AND d.value_stddev = 0, false) as is_flatline,
            -- Latest reading up to and including this day (running max per sensor)
            MAX(d.last_reading_utc) OVER (
                PARTITION BY g.sensor_id ORDER BY g.date_id
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) as last_reading_so_far
        FROM grid g
        LEFT JOIN daily d ON g.sensor_id = d.sensors_id AND g.date_id = d.date_id
    )

    SELECT
        sensor_id,
        sensor_name,
        parameter_name,
        location_name,
        locality,
        date_id,
        readings,
        is_flatline,
        -- Status measured at each day's end (midnight of the following day)
        CASE
            WHEN last_reading_so_far IS NULL THEN 'NEVER_REPORTED'
            WHEN TIMESTAMPDIFF(HOUR, last_reading_so_far, date_id + INTERVAL 1 DAY) >= {DOWN_AFTER_HOURS} THEN 'DOWN'
            WHEN TIMESTAMPDIFF(HOUR, last_reading_so_far, date_id + INTERVAL 1 DAY) >= {STALE_AFTER_HOURS} THEN 'STALE'
            ELSE 'HEALTHY'
        END as status
    FROM filled
""")

# Full rebuild on every run: the table grows one row per sensor per day, so
# rebuilding stays fast for years and is idempotent by construction.
df_sensor_health_daily.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.sensor_health_daily")
print(f"sensor_health_daily: {spark.table(f'{CATALOG}.gold.sensor_health_daily').count()} sensor-days")

### 5.0: Verify

In [ ]:
# Sanity check: current sensors by status, and the daily status trend
print("Current state")
spark.sql(f"""
    SELECT status, COUNT(*) as sensors, SUM(CAST(needs_maintenance AS INT)) as flagged
    FROM {CATALOG}.gold.sensor_health
    GROUP BY status
    ORDER BY sensors DESC
""").display()

print("Daily trend (last 10 days)")
spark.sql(f"""
    SELECT date_id, status, COUNT(*) as sensors
    FROM {CATALOG}.gold.sensor_health_daily
    WHERE date_id >= (SELECT MAX(date_id) - INTERVAL 9 DAYS FROM {CATALOG}.gold.sensor_health_daily)
    GROUP BY date_id, status
    ORDER BY date_id, status
""").display()